# Common CM-blind optimizer harness

This notebook validates the first stage of the adversarial search program for $g=2$, $D=(1,5)$. Every future optimizer will receive the same six-coordinate compatible-metric objective, while CM reconstruction remains completely outside the optimization loop.

In [1]:
from dataclasses import asdict
from math import sqrt
from pathlib import Path
from tempfile import TemporaryDirectory
import sys
import numpy as np

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from gkp_systole import (
    EvaluationMode, OptimizationRunConfig, OptimizerHarness,
    type15_compatible_metric_family,
)

In [2]:
family = type15_compatible_metric_family()
config = OptimizationRunConfig.symmetric_box(
    experiment_id='type15-notebook-check',
    optimizer_name='deterministic-random-baseline',
    seed=20260715,
    coordinate_dimension=family.coordinate_dimension,
    radius=0.25,
    evaluation_budget=40,
    verification_decimal_places=60,
)
harness = OptimizerHarness(family, config)
origin = harness.evaluate((0.0,) * family.coordinate_dimension)
{
    'family': family.name,
    'type': family.polarization.type,
    'coordinate_dimension': family.coordinate_dimension,
    'ell_squared': origin.squared_systole,
    'exact_record': family.reference_exact_ell_squared,
    'classes/lifts': (origin.class_multiplicity, origin.lift_multiplicity),
    'diagnostics': asdict(origin.diagnostics),
}

{'family': 'type (1,5) exact-record compatible chart',
 'type': (1, 5),
 'coordinate_dimension': 6,
 'ell_squared': 0.6324555320336387,
 'exact_record': '2/sqrt(10) = sqrt(2/5)',
 'classes/lifts': (24, 24),
 'diagnostics': {'minimum_eigenvalue': 0.02393378128158209,
  'symmetry_residual': 0.0,
  'determinant_residual': 4.227729277772596e-13,
  'complex_structure_residual': 6.821210263296962e-13,
  'valid': True}}

In [3]:
random = np.random.default_rng(config.seed)
coordinates = random.uniform(-0.05, 0.05, size=(24, family.coordinate_dimension))
samples = harness.evaluate_many(coordinates, workers=2)
top = sorted(samples, key=lambda sample: sample.squared_systole, reverse=True)[:5]
{
    'top_random_values': [sample.squared_systole for sample in top],
    'origin_record': origin.squared_systole,
    'summary': asdict(harness.summary()),
}

{'top_random_values': [0.6102658965361751,
  0.6101407604990792,
  0.6014713131070408,
  0.6013100893365078,
  0.600700366034083],
 'origin_record': 0.6324555320336387,
 'summary': {'request_count': 25,
  'unique_evaluation_count': 25,
  'screening_evaluation_count': 25,
  'cache_hits': 0,
  'best_squared_systole': 0.6324555320336387,
  'best_coordinates': (0.0, 0.0, 0.0, 0.0, 0.0, 0.0)}}

In [4]:
verified = harness.evaluate(origin.coordinates, mode=EvaluationMode.VERIFY)
with TemporaryDirectory() as directory:
    checkpoint = harness.save_checkpoint(Path(directory) / 'type15.jsonl')
    resumed = OptimizerHarness.from_checkpoint(family, checkpoint)
    resumed_summary = asdict(resumed.summary())
{
    'screen_value': origin.squared_systole,
    'high_precision_value': verified.high_precision_squared_systole,
    'checkpoint_resume': resumed_summary,
}

{'screen_value': 0.6324555320336387,
 'high_precision_value': '0.63245553203354920000000000000000000000000000000000000000002',
 'checkpoint_resume': {'request_count': 26,
  'unique_evaluation_count': 26,
  'screening_evaluation_count': 25,
  'cache_hits': 0,
  'best_squared_systole': 0.6324555320336387,
  'best_coordinates': (0.0, 0.0, 0.0, 0.0, 0.0, 0.0)}}

In [5]:
assert family.coordinate_dimension == 6
assert family.polarization.type == (1, 5)
assert abs(origin.squared_systole - sqrt(2/5)) < 1e-12
assert origin.class_multiplicity == origin.lift_multiplicity == 24
assert origin.diagnostics.valid
assert abs(float(verified.high_precision_squared_systole) - origin.squared_systole) < 1e-12
'All optimizer-harness checks passed.'

'All optimizer-harness checks passed.'